# Video-LLaVA / MuLER Stage 1 Image Captioning Benchmark (COCO Karpathy Test Split)

Unlike VQA benchmarks (like POPE) that require Stage 2 instruction tuning, **COCO Image Captioning (Karpathy Test Split)** directly measures **Stage 1 pretraining quality** (visual alignment, grounding, fluency, and mode-collapse detection).

### Models Compared
1. **Untrained (Step 0)** — `lmsys/vicuna-7b-v1.5` + random `mm_projector` + zeroed Gate v2
2. **Stage 1 Projector-Only (No Fusion)** — frozen official `mm_projector` from `checkpoint-800` with Gate v2 forced to `0` *(isolates the exact effect of your early-fusion MHA)*
3. **`checkpoint-800` (Stage 1, step 800)** — healthy pre-spike checkpoint (`loss=1.827`)
4. **`final` (Stage 1, step 867)** — post-spike checkpoint (`loss=4.950`)
5. **`checkpoint-4750` (Stage 1, step 4750)** — collapsed checkpoint (`loss=5.697`)
6. *(Optional)* **Official `LanguageBind/Video-LLaVA-7B`** — Stage 2 instruction-tuned upper reference

### Metrics
- **CIDEr** (primary COCO captioning metric; higher is better)
- **BLEU-1**, **BLEU-4**, **ROUGE-L**
- **Distinct-1**, **Repetition %**, **Empty %** (detects collapse like `'aЪЪЪЪ...'` or `'skier skier'`)

### Estimated Runtime on Colab L4 (24 GB, `bf16`)
- **50-image quick smoke test (all 5 Stage 1 arms):** ~2–3 min total
- **500-image full benchmark (per model):** ~5–8 min each

## 1. Check GPU & Auto-Selected Precision

In [ ]:
!nvidia-smi
import torch
if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    prec = '4bit-fp16' if vram < 20 else ('bf16' if torch.cuda.is_bf16_supported() else 'fp16')
    print(f"GPU: {name} | VRAM: {vram:.1f} GB | Auto precision: {prec}")
else:
    print("WARNING: No CUDA GPU detected! Switch Runtime -> Change runtime type -> GPU.")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_EVAL_OUT = "/content/drive/MyDrive/Video-LLaVA/eval_results/coco_caption"
os.makedirs(DRIVE_EVAL_OUT, exist_ok=True)
print(f"Results will be synced to: {DRIVE_EVAL_OUT}")

## 3. Clone / Pull Repository & Install Dependencies

In [ ]:
%cd /content
![ ! -d "grad_project" ] && git clone https://github.com/davidrimon2004/grad_project
%cd /content/grad_project
!git pull

!pip install -q --upgrade pip
!pip install -q transformers tokenizers sentencepiece shortuuid accelerate peft bitsandbytes einops einops-exts timm huggingface_hub gdown matplotlib tqdm
!pip install -q -e .

## 4. Prepare COCO Karpathy Test Split (500 Images, 5 Human Captions Each)
Reuses any cached COCO `val2014` images from `/content/eval/pope/val2014` if present.

In [ ]:
!python scripts/eval_caption_colab.py \
    --action prepare \
    --eval_dir /content/eval/coco_caption \
    --num_images 500 \
    --reuse_val2014_dir /content/eval/pope/val2014

## 5. Fetch Checkpoints (`checkpoint-800`, `final`, `checkpoint-4750`, and `untrained_step0`)

In [ ]:
!python scripts/eval_caption_colab.py \
    --action download_ckpt \
    --ckpt_name all \
    --drive_ckpt_root /content/drive/MyDrive/Video-LLaVA/checkpoints/videollava-7b-pretrain \
    --local_ckpt_root /content/checkpoints/videollava-7b-pretrain

## 6. Quick 50-Image Smoke Test Across All 5 Stage 1 Arms (~2–3 min on L4)
Runs 50 Karpathy test images per arm and displays the CIDEr / BLEU / ROUGE-L table + sample captions.

In [ ]:
%%bash
set -e
python scripts/eval_caption_colab.py --action eval --untrained \
    --model_path /content/checkpoints/videollava-7b-pretrain/untrained_step0 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/smoke_untrained.jsonl \
    --metrics_file /content/eval/coco_caption/results/smoke_untrained_metrics.json \
    --max_samples 50 --precision auto --label "Untrained (Step 0)"

python scripts/eval_caption_colab.py --action eval --zero_gate \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-800 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/smoke_proj_only.jsonl \
    --metrics_file /content/eval/coco_caption/results/smoke_proj_only_metrics.json \
    --max_samples 50 --precision auto --label "Stage 1 Projector-Only (No Fusion)"

python scripts/eval_caption_colab.py --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-800 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/smoke_ckpt800.jsonl \
    --metrics_file /content/eval/coco_caption/results/smoke_ckpt800_metrics.json \
    --max_samples 50 --precision auto --label "checkpoint-800 (Stage 1, step 800)"

python scripts/eval_caption_colab.py --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/final \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/smoke_final.jsonl \
    --metrics_file /content/eval/coco_caption/results/smoke_final_metrics.json \
    --max_samples 50 --precision auto --label "final (Stage 1, step 867)"

python scripts/eval_caption_colab.py --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-4750 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/smoke_ckpt4750.jsonl \
    --metrics_file /content/eval/coco_caption/results/smoke_ckpt4750_metrics.json \
    --max_samples 50 --precision auto --label "checkpoint-4750 (Stage 1, step 4750)"

python scripts/eval_caption_colab.py --action compare \
    --metrics_files \
      /content/eval/coco_caption/results/smoke_untrained_metrics.json \
      /content/eval/coco_caption/results/smoke_proj_only_metrics.json \
      /content/eval/coco_caption/results/smoke_ckpt800_metrics.json \
      /content/eval/coco_caption/results/smoke_final_metrics.json \
      /content/eval/coco_caption/results/smoke_ckpt4750_metrics.json \
    --output_dir /content/eval/coco_caption/results/smoke

## 7A. Full 500-Image Eval: Untrained (Step 0)

In [ ]:
!python scripts/eval_caption_colab.py \
    --action eval --untrained \
    --model_path /content/checkpoints/videollava-7b-pretrain/untrained_step0 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/cap_untrained.jsonl \
    --metrics_file /content/eval/coco_caption/results/cap_untrained_metrics.json \
    --precision auto \
    --label "Untrained (Step 0)"

## 7B. Full 500-Image Eval: Stage 1 Projector-Only (No Early Fusion)

In [ ]:
!python scripts/eval_caption_colab.py \
    --action eval --zero_gate \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-800 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/cap_proj_only.jsonl \
    --metrics_file /content/eval/coco_caption/results/cap_proj_only_metrics.json \
    --precision auto \
    --label "Stage 1 Projector-Only (No Fusion)"

## 7C. Full 500-Image Eval: `checkpoint-800` (Pre-Spike, Healthy)

In [ ]:
!python scripts/eval_caption_colab.py \
    --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-800 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/cap_ckpt800.jsonl \
    --metrics_file /content/eval/coco_caption/results/cap_ckpt800_metrics.json \
    --precision auto \
    --label "checkpoint-800 (Stage 1, step 800)"

## 7D. Full 500-Image Eval: `final` (Step 867)

In [ ]:
!python scripts/eval_caption_colab.py \
    --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/final \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/cap_final.jsonl \
    --metrics_file /content/eval/coco_caption/results/cap_final_metrics.json \
    --precision auto \
    --label "final (Stage 1, step 867)"

## 7E. (Optional) Full 500-Image Eval: `checkpoint-4750` (Collapsed)

In [ ]:
!python scripts/eval_caption_colab.py \
    --action eval \
    --model_path /content/checkpoints/videollava-7b-pretrain/checkpoint-4750 \
    --model_base lmsys/vicuna-7b-v1.5 \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/cap_ckpt4750.jsonl \
    --metrics_file /content/eval/coco_caption/results/cap_ckpt4750_metrics.json \
    --precision auto \
    --label "checkpoint-4750 (Stage 1, step 4750)"

## 8. (Optional) Official `LanguageBind/Video-LLaVA-7B` Reference

In [ ]:
!python scripts/eval_caption_colab.py \
    --action eval \
    --model_path LanguageBind/Video-LLaVA-7B \
    --eval_dir /content/eval/coco_caption \
    --answers_file /content/eval/coco_caption/answers/cap_official_ref.jsonl \
    --metrics_file /content/eval/coco_caption/results/cap_official_ref_metrics.json \
    --precision auto \
    --label "Official Video-LLaVA-7B (Stage 2 Ref)"

## 9. Comparison Table, Chart & Google Drive Sync

In [ ]:
import os, json, shutil, subprocess
from IPython.display import display, Markdown
import matplotlib.pyplot as plt
import numpy as np

candidate_files = [
    '/content/eval/coco_caption/results/cap_untrained_metrics.json',
    '/content/eval/coco_caption/results/cap_proj_only_metrics.json',
    '/content/eval/coco_caption/results/cap_ckpt800_metrics.json',
    '/content/eval/coco_caption/results/cap_final_metrics.json',
    '/content/eval/coco_caption/results/cap_ckpt4750_metrics.json',
    '/content/eval/coco_caption/results/cap_official_ref_metrics.json',
]
existing_files = [p for p in candidate_files if os.path.exists(p)]
assert len(existing_files) >= 2, f'Run at least 2 eval cells first; found: {existing_files}'

subprocess.run([
    'python', 'scripts/eval_caption_colab.py', '--action', 'compare',
    '--metrics_files', *existing_files,
    '--output_dir', '/content/eval/coco_caption/results',
    '--drive_output_dir', '/content/drive/MyDrive/Video-LLaVA/eval_results/coco_caption',
], check=True)

display(Markdown(open('/content/eval/coco_caption/results/caption_comparison_report.md').read()))

with open('/content/eval/coco_caption/results/caption_comparison_results.json') as f:
    data = json.load(f)

models = data['models']
names = [m['name'] for m in models]
x = np.arange(len(names))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, key, title in zip(axes, ['cider', 'bleu_4', 'rouge_l'], ['CIDEr', 'BLEU-4', 'ROUGE-L']):
    vals = [m['metrics'][key] for m in models]
    bars = ax.bar(x, vals)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=25, ha='right', fontsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1f}', (bar.get_x() + bar.get_width()/2, h), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
chart_path = '/content/eval/coco_caption/results/caption_comparison_chart.png'
plt.savefig(chart_path, dpi=150)
plt.show()

if os.path.exists('/content/drive/MyDrive/Video-LLaVA/eval_results/coco_caption'):
    shutil.copy2(chart_path, '/content/drive/MyDrive/Video-LLaVA/eval_results/coco_caption/caption_comparison_chart.png')
    print('Chart saved to Google Drive.')